# Notebook 05 — Diagnose validation errors before the next training experiment

The first adapter learned better answer formatting, but word-problem accuracy did not consistently improve. This notebook investigates that result using **50 fixed original validation questions**, the original base model, and the selected first adapter.

Each model answers each question with zero-shot and three-shot prompts: **200 predictions total**. No new training or test-benchmark evaluation runs. The official v2 stopping/scoring rules remain fixed.

**Kaggle:** enable Internet, select a T4, and attach only:

1. **Original Notebook 01 saved output** — data bundle.
2. **Notebook 03 saved output** — full-run selected adapter and manifests.

Notebook 02c and 04 outputs are not required. Questions were excluded from gradient training, but the full validation split was used for validation-loss checkpoint selection. These results are development diagnosis, not an untouched final holdout.


## 1. Fetch shared code

The exact GitHub commit is recorded. Keep the original data bundle and first adapter unchanged.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import sys
import tempfile

REPOSITORY = "https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git"
PROJECT_REF = os.environ.get("MATH_PROJECT_REF", "main")
if PROJECT_REF.startswith("-"):
    raise ValueError("Use a branch, tag, or full commit SHA as MATH_PROJECT_REF")
PROJECT = Path(tempfile.mkdtemp(prefix="math-project-"))
subprocess.run(["git", "init", "--quiet", str(PROJECT)], check=True)
subprocess.run(["git", "-C", str(PROJECT), "remote", "add", "origin", REPOSITORY], check=True)
subprocess.run(["git", "-C", str(PROJECT), "fetch", "--depth", "1", "origin", PROJECT_REF], check=True)
subprocess.run(["git", "-C", str(PROJECT), "checkout", "--detach", "--quiet", "FETCH_HEAD"], check=True)
CODE_COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("Repository:", REPOSITORY)
print("Code commit:", CODE_COMMIT)
print("Project checkout:", PROJECT)

Repository: https://github.com/sahan24854/Fine-Tuning-LLM-for-math.git
Code commit: 1ebdf5824cdf391f689cd0211a052f2fef591d9e
Project checkout: /tmp/math-project-gg5f9gwu


From https://github.com/sahan24854/Fine-Tuning-LLM-for-math
 * branch            main       -> FETCH_HEAD
 * [new branch]      main       -> origin/main


## 2. Find inputs and verify the diagnostic scope

The sample is selected by SHA-256 ranking of seed and original validation ID, independently of answers or model correctness. The same 50 questions are used in both modes and for both models.

Three-shot uses the original frozen training demonstrations. Zero-shot provides instructions and the problem only. Reference validation solutions are displayed later for review and are never supplied to the generator.


In [2]:
import hashlib
import shutil

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Enable a GPU accelerator in Kaggle settings.")
_ = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=True)
BUNDLE = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_bundle.py"), os.environ.get("MATH_BUNDLE_PATH", "/kaggle/input")], text=True).strip())
TRAINING = Path(subprocess.check_output([sys.executable, str(PROJECT / "scripts/find_results.py"), os.environ.get("MATH_TRAINING_PATH", "/kaggle/input"), "--kind", "training"], text=True).strip())
CONFIG_PATH = PROJECT / "configs/validation_analysis.json"
CONFIG_HASH = hashlib.sha256(CONFIG_PATH.read_bytes()).hexdigest()
verified = subprocess.check_output([sys.executable, str(PROJECT / "scripts/analyze_validation.py"),
    "--bundle", str(BUNDLE), "--training", str(TRAINING), "--config", str(CONFIG_PATH),
    "--output", "/kaggle/working/validation-verification-only", "--stage", "verify"], text=True, cwd=PROJECT)
VERIFIED = json.loads(verified)
OUTPUT = Path("/kaggle/working") / f"validation-analysis-{VERIFIED['bundle_sha256'][:8]}-{VERIFIED['training_sha256'][:8]}-{CONFIG_HASH[:8]}-{CODE_COMMIT[:8]}"
print("Data:", BUNDLE)
print("Training:", TRAINING)
print(json.dumps(VERIFIED, indent=2))
print("Output:", OUTPUT)


Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04
Data: /kaggle/input/notebooks/sahan24854/01-data-and-protocol/data-protocol-20261019T082909Z-1cb02106
Training: /kaggle/input/notebooks/sahan24854/03-teach-smollm2-maths-with-lora/lora-v1-b9d3419aa714-1100d832-25f43858/full
{
  "questions": 50,
  "predictions_per_model": 100,
  "total_predictions": 200,
  "bundle_sha256": "b9d3419aa71419626d274843f88dc6190f5828877b20a1cc67e5249e760b5c71",
  "training_sha256": "c1a577f42d943230dc125ddd894fda6f6dd686505fe64196fe068935b95b3063",
  "checkpoint": "checkpoint-633",
  "interpretation": "These questions were excluded from gradient training but used for validation-loss checkpoint selection. This is development diagnosis, not an untouched holdout."
}
Output: /kaggle/working/validation-analysis-b9d3419a-c1a577f4-7f2ee8cd-1ebdf582


## 3. Prepare the known inference/adapter environment

Use one T4 with the pinned PyTorch, Transformers, and PEFT versions. Both models run sequentially in separate processes to release GPU memory between them. Generation is greedy, batch four, with the reviewed 512-token cap and per-row stopping.

The existing nonfatal `wrapt` startup and `torch_dtype` deprecation warnings may appear. CUDA execution and CPU contract tests must still pass.


In [3]:
ENVIRONMENT = Path(tempfile.mkdtemp(prefix="math-validation-env-"))
_ = subprocess.run([sys.executable, str(PROJECT / "scripts/setup_environment.py"),
                    "--environment", str(ENVIRONMENT), "--profile", "training"], check=True)
PYTHON = ENVIRONMENT / "bin" / "python"
INFERENCE_ENV = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
probe = """
import torch, peft
if not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable.')
x = torch.ones((32, 32), device='cuda', dtype=torch.float16)
assert (x @ x).sum().item() == 32768
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), peft.__version__)
"""
_ = subprocess.run([str(PYTHON), "-I", "-c", probe], check=True, env=INFERENCE_ENV)


Creating isolated environment with Python 3.13.15


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Processing /tmp/math-pip-bootstrap-5b8izfow/pip-25.3-py3-none-any.whl


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 95.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 65.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 111.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 134.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 17.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 29.4 MB/s  0:00:00



...

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


No broken requirements found.
Environment ready: /tmp/math-validation-env-34gr1q36/bin/python


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


2.7.1+cu118 11.8 Tesla T4 0.17.1


## 4. Run CPU contract tests

Tiny local models/adapters check validation-only scope, deterministic selection, literal-equality flags, identity protection, interruption recovery, and both prompt modes. These tests do not estimate SmolLM2 accuracy.


In [4]:
_ = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", "test_validation_analysis.py", "-b", "-v"],
                    cwd=PROJECT, check=True, env=INFERENCE_ENV)


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
test_flags_only_whole_literal_equalities_and_never_execute_expressions (test_validation_analysis.ValidationAnalysisTests.test_flags_only_whole_literal_equalities_and_never_execute_expressions) ... ok
test_paired_counts_align_keys_instead_of_position (test_validation_analysis.ValidationAnalysisTests.test_paired_counts_align_keys_instead_of_position) ... ok
test_selection_is_bounded_validation_only_and_independent_of_order_or_correctness (test_validation_analysis.ValidationAnalysisTests.test_selection_is_bounded_validation_only_and_independent_of_order_or_correctness) ... ok
test_interruption_resume_report_identity_and_review_template (test_validation_analysis.ValidationPipelineTests.test_interruption_resume_report_identity_and_review_template) ... ok
test_real_cli_both_modes_both_models_and_no_test_questions (test_validation_analysis.ValidationPipelineTests.test_real_cli_both_modes_both

## 5. Generate original-model answers — 100 predictions

Generate 50 zero-shot and 50 three-shot answers. Save each batch so an interrupted stage can resume in the same environment. Changed inputs, code, or runtime are refused. Original data and adapter inputs remain read-only.


In [5]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/analyze_validation.py"),
    "--bundle", str(BUNDLE), "--training", str(TRAINING), "--config", str(CONFIG_PATH),
    "--output", str(OUTPUT), "--stage", "base"], check=True, cwd=PROJECT, env=INFERENCE_ENV)


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Loading HuggingFaceTB/SmolLM2-360M at f8027fd0eaeea54caa13c31d31b9fdc459c38b49 on Tesla T4


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded in 3.9s
Validation base: 0/100 predictions saved
Saved 4/100 base validation predictions
Saved 8/100 base validation predictions
Saved 12/100 base validation predictions
Saved 16/100 base validation predictions
Saved 20/100 base validation predictions
Saved 24/100 base validation predictions
Saved 28/100 base validation predictions
Saved 32/100 base validation predictions
Saved 36/100 base validation predictions
Saved 40/100 base validation predictions
Saved 44/100 base validation predictions
Saved 48/100 base validation predictions
Saved 52/100 base validation predictions
Saved 56/100 base validation predictions
Saved 60/100 base validation predictions
Saved 64/100 base validation predictions
Saved 68/100 base validation predictions
Saved 72/100 base validation predictions
Saved 76/100 base validation predictions
Saved 80/100 base validation predictions
Saved 84/100 base validation predictions
Saved 88/100 base validation predictions
Saved 92/100 base validation predictio

## 6. Generate selected-adapter answers — 100 predictions

Use the same questions, prompts, stopping, and scoring. The original model's core runtime must match. This stage loads the full selected first adapter without changing its weights.


In [6]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/analyze_validation.py"),
    "--bundle", str(BUNDLE), "--training", str(TRAINING), "--config", str(CONFIG_PATH),
    "--output", str(OUTPUT), "--stage", "tuned"], check=True, cwd=PROJECT, env=INFERENCE_ENV)


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


Loading HuggingFaceTB/SmolLM2-360M at f8027fd0eaeea54caa13c31d31b9fdc459c38b49 on Tesla T4


`torch_dtype` is deprecated! Use `dtype` instead!


Model loaded in 1.2s
Validation tuned: 0/100 predictions saved
Saved 4/100 tuned validation predictions
Saved 8/100 tuned validation predictions
Saved 12/100 tuned validation predictions
Saved 16/100 tuned validation predictions
Saved 20/100 tuned validation predictions
Saved 24/100 tuned validation predictions
Saved 28/100 tuned validation predictions
Saved 32/100 tuned validation predictions
Saved 36/100 tuned validation predictions
Saved 40/100 tuned validation predictions
Saved 44/100 tuned validation predictions
Saved 48/100 tuned validation predictions
Saved 52/100 tuned validation predictions
Saved 56/100 tuned validation predictions
Saved 60/100 tuned validation predictions
Saved 64/100 tuned validation predictions
Saved 68/100 tuned validation predictions
Saved 72/100 tuned validation predictions
Saved 76/100 tuned validation predictions
Saved 80/100 tuned validation predictions
Saved 84/100 tuned validation predictions
Saved 88/100 tuned validation predictions
Saved 92/100 tu

## 7. Review failures and save the diagnostic

The report separates accuracy, parseability, cap hits, repetition, and literal numeric-equality flags for all four model/mode combinations. Expand paired answers beside the reference worked solution. Inspect improvements and regressions.

The arithmetic checker recognizes only a whole line such as `52 + 18 = 60`. It never executes arbitrary expressions or repairs the answer. A mismatch is a review flag: it may involve rounding, a quoted calculation, or an actual arithmetic error. Narrative calculations, variables, and more complex expressions remain unchecked. Correct final answers can still have flawed reasoning.

Use a **copy** of `review_template.csv` to annotate wrong operations/quantities, arithmetic errors, format failures, repetitive nonanswers, reasoning concerns, or ambiguous cases. Automatic flags do not assign reasoning-error causes.

This notebook ends with review evidence. Choose the next controlled training change after inspecting validation failures. Keep the original test results fixed; do not tune to the 300 inspected pilot questions.


In [7]:
_ = subprocess.run([str(PYTHON), str(PROJECT / "scripts/analyze_validation.py"),
    "--bundle", str(BUNDLE), "--training", str(TRAINING), "--config", str(CONFIG_PATH),
    "--output", str(OUTPUT), "--stage", "report"], check=True, cwd=PROJECT, env=INFERENCE_ENV)
SUMMARY = json.loads((OUTPUT / "summary.json").read_text())
RESULT = json.loads((OUTPUT / "result_manifest.json").read_text())
for name, digest in RESULT["files_sha256"].items():
    if hashlib.sha256((OUTPUT / name).read_bytes()).hexdigest() != digest:
        raise RuntimeError(f"Validation artifact checksum mismatch: {name}")
from IPython.display import HTML, display
display(HTML((OUTPUT / "report.html").read_text()))
print(json.dumps(SUMMARY, indent=2))
ARCHIVE = Path(shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT))
print("Diagnostic ZIP:", ARCHIVE)
print("Save Version with outputs. Share summary.json and paired_predictions.jsonl, or the executed notebook.")


Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'


{
  "stage": "validation_error_analysis",
  "analysis_id": "gsm8k-validation-error-analysis-1",
  "execution_complete": true,
  "questions": 50,
  "predictions": 200,
  "expected_predictions": 200,
  "groups": {
    "base/zero_shot": {
      "count": 50,
      "expected": 50,
      "correct": 0,
      "accuracy": 0.0,
      "accuracy_wilson_95": [
        6.938893903907228e-18,
        0.07134759913335872
      ],
      "parseable": 3,
      "parse_rate": 0.06,
      "strict_numeric_format_rate": 0.02,
      "invalid": 47,
      "cap_hits": 13,
      "stop_reasons": {
        "next_question": 33,
        "length": 13,
        "final_answer_line": 4
      },
      "generated_tokens": 9923,
      "mean_generated_tokens": 198.46,
      "generation_seconds": 322.13655191600014,
      "repetition_flagged": 17,
      "numeric_equality_review_flagged": 0
    },
    "base/few_shot": {
      "count": 50,
      "expected": 50,
      "correct": 5,
      "accuracy": 0.1,
      "accuracy_wilson_95"

Model / mode,Count,Accuracy,Parse rate,Cap hits,Repetition,Equality review flags
base/zero_shot,50/50,0.0%,6.0%,13,17,0
base/few_shot,50/50,10.0%,62.0%,9,9,0
tuned/zero_shot,50/50,4.0%,88.0%,6,3,0
tuned/few_shot,50/50,12.0%,96.0%,2,2,0


{
  "stage": "validation_error_analysis",
  "analysis_id": "gsm8k-validation-error-analysis-1",
  "execution_complete": true,
  "questions": 50,
  "predictions": 200,
  "expected_predictions": 200,
  "groups": {
    "base/zero_shot": {
      "count": 50,
      "expected": 50,
      "correct": 0,
      "accuracy": 0.0,
      "accuracy_wilson_95": [
        6.938893903907228e-18,
        0.07134759913335872
      ],
      "parseable": 3,
      "parse_rate": 0.06,
      "strict_numeric_format_rate": 0.02,
      "invalid": 47,
      "cap_hits": 13,
      "stop_reasons": {
        "next_question": 33,
        "length": 13,
        "final_answer_line": 4
      },
      "generated_tokens": 9923,
      "mean_generated_tokens": 198.46,
      "generation_seconds": 322.13655191600014,
      "repetition_flagged": 17,
      "numeric_equality_review_flagged": 0
    },
    "base/few_shot": {
      "count": 50,
      "expected": 50,
      "correct": 5,
      "accuracy": 0.1,
      "accuracy_wilson_95"